In [ ]:
import pandas as pd
import re
import contractions

def preprocessing(text):
    if type(text) != str:
        return ""
    
    text = text.lower()
    
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'www\.\S+', '', text)
    
    text = contractions.fix(text)
    
    text = " ".join(text.split())
    
    return text

In [ ]:
from transformers import pipeline

sentiment=pipeline(
    'text-classification',
    model='./model_new/checkpoint-10000',
    tokenizer='distilbert-base-uncased'
)

test = [
    "The food was absolutely delicious and the delivery was fast!",
    "I will never buy this product again. It tastes like garbage and the box was broken."
]

label_map={
    'LABEL_1':'Positive',
    'LABEL_0':'Negative'
}

predict=sentiment(test)
for text,pred in zip(test,predict):
    label=label_map.get(pred['label'],pred['label'])
    print(f'Teks: {text}')
    print(f"Prediksi: {label} (Accuracy: {pred['score']})\n")

In [ ]:
import spacy
nlp=spacy.load('en_core_web_sm')

def output(text):
    clean_text = preprocessing(text)
    doc = nlp(clean_text)
    hasil_ekstraksi = []

    for token in doc:
        # ======================================================
        # SKENARIO 1 & 2: Opini berupa Kata Sifat (ADJ)
        # ======================================================
        if token.pos_ == "ADJ":
            aspect = None
            opinion = token.text
            verb_context = "" 
            intensifier = "" 
            
            for child in token.children:
                if child.dep_ == "advmod":
                    intensifier = child.text + " "
            
            if token.dep_ == "amod" and token.head.pos_ == "NOUN":
                aspect = token.head.text
                
            elif token.dep_ == "acomp" and token.head.pos_ in ["AUX", "VERB"]:
                for child in token.head.children:
                    if child.dep_ == "nsubj" and child.pos_ == "NOUN":
                        aspect = child.text
                        break
            
            if aspect:
                for ancestor in token.ancestors:
                    if ancestor.pos_ == "VERB":
                        verb = ancestor.text
                        subj = ""
                        verb_neg = ""
                        for child in ancestor.children:
                            if child.dep_ in ["nsubj", "nsubjpass", "PRON"]:
                                subj = child.text + " "
                            if child.dep_ == "neg":
                                verb_neg = child.text + " "
                        verb_context = f"{subj}{verb_neg}{verb} "
                        break 
        
            if aspect and opinion:
                negation = ""
                for child in token.children:
                    if child.dep_ == "neg":
                        negation = child.text + " "
                if token.head.pos_ in ["AUX", "VERB"]:
                    for child in token.head.children:
                        if child.dep_ == "neg":
                            negation = child.text + " "

                opini_lengkap = f"{negation}{intensifier}{opinion}".strip()
                
                if verb_context:
                    kalimat_fokus = f"{verb_context}the {aspect} is {opini_lengkap}"
                else:
                    kalimat_fokus = f"The {aspect} is {opini_lengkap}"
                
                kalimat_fokus = " ".join(kalimat_fokus.split())
                
                prediksi_sentimen = sentiment(kalimat_fokus)[0]
                label = "Positif" if prediksi_sentimen['label'] == 'LABEL_1' else "Negatif"
                skor = f"{prediksi_sentimen['score'] * 100:.1f}%"
                
                hasil_ekstraksi.append({
                    "Aspek": aspect.capitalize(),
                    "Opini Pengguna": opini_lengkap.lower(),
                    "Kalimat Evaluasi": kalimat_fokus, 
                    "Sentimen": label,
                    "Tingkat Keyakinan": skor
                })

        # ======================================================
        # SKENARIO 3 (BARU): Opini berupa Kata Kerja (VERB)
        # Contoh kalimat: "I love the cake and coffee and bread"
        # ======================================================
        elif token.pos_ == "VERB":
            opinion = token.text
            
            # Cek negasi (misal: "don't love")
            negation = ""
            for child in token.children:
                if child.dep_ == "neg":
                    negation = child.text + " "
                    
            # Cari subjek yang melakukan kata kerja
            subj = ""
            for child in token.children:
                if child.dep_ in ["nsubj", "PRON"]:
                    subj = child.text + " "
                    
            # Cari objek utama makanan yang dikenai kata kerja (dobj)
            for child in token.children:
                if child.dep_ == "dobj" and child.pos_ == "NOUN":
                    # Makanan pertama ditemukan (contoh: "cake")
                    daftar_makanan = [child]
                    
                    # MAGIS SPACY: Menangkap makanan lain yang dirangkai dengan "and"
                    for conj in child.conjuncts:
                        if conj.pos_ == "NOUN":
                            daftar_makanan.append(conj)
                            
                    # Proses seluruh makanan yang ditemukan
                    for makanan_token in daftar_makanan:
                        aspect = makanan_token.text
                        
                        # Rakit kalimat, contoh hasil: "i love the cake"
                        kalimat_fokus = f"{subj}{negation}{opinion} the {aspect}"
                        kalimat_fokus = " ".join(kalimat_fokus.split())
                        
                        prediksi_sentimen = sentiment(kalimat_fokus)[0]
                        label = "Positif" if prediksi_sentimen['label'] == 'LABEL_1' else "Negatif"
                        skor = f"{prediksi_sentimen['score'] * 100:.1f}%"
                        
                        hasil_ekstraksi.append({
                            "Aspek": aspect.capitalize(),
                            "Opini Pengguna": f"{negation}{opinion}".strip().lower(),
                            "Kalimat Evaluasi": kalimat_fokus,
                            "Sentimen": label,
                            "Tingkat Keyakinan": skor
                        })
                    break # Berhenti mencari jika objek sudah ketemu

    if not hasil_ekstraksi:
        return pd.DataFrame([{"Pesan": "Tidak ada aspek dan opini yang berhasil diekstrak dari kalimat."}])
    
    return pd.DataFrame(hasil_ekstraksi)

In [ ]:
# ==========================================
# Uji Coba Output Terstruktur
# ==========================================
ulasan_user = "Overall, Emilia Cucina Italiana is a wonderful place not only for great food, but also for outstanding hospitality and special moments. I would highly recommend it and would be happy to return."
df_hasil = output(ulasan_user)

# Jika dijalankan di VS Code/Jupyter, ini akan mencetak tabel yang sangat rapi
display(df_hasil)